# 04 — Рейтинг и ковенанты
Кредитный анализ: Base / Forecast / Stress рейтинги + covenant breach impact.

**Разделы:**
1. Настройка (build_model с run_stress/rating/covenants=True)
2. Кредитные метрики
3. Base / Forecast / Stress рейтинги
4. Ковенанты (base vs stress)
5. Covenant Breach — влияние на рейтинг
6. Итоговая таблица


## §1 Настройка

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
ROOT = NOTEBOOK_DIR.parents[2]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

COMPANY_ID    = 'us_steel'
SCENARIO_NAME = 'base'
DB_PATH       = ROOT / 'data_mart_v2.db'
CONFIG_PATH   = ROOT / 'companies' / COMPANY_ID / 'configs' / 'project.yaml'

import logging, warnings, pandas as pd
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

from engine.orchestrator import build_model
from engine.database.repository import Repository
from engine.model.loader import ModelInputLoader
from engine.rating.runner import RatingRunner
from engine.covenants.core import CovenantsChecker

# Build model with all features enabled
result = build_model(
    company_id=COMPANY_ID,
    scenario_name=SCENARIO_NAME,
    db_path=DB_PATH,
    config_path=CONFIG_PATH,
    run_preprocessor=False,
    run_macro=False,
    run_model=True,
    run_stress=True,
    run_rating=True,
    run_covenants=True,
)
mr = result.model_result
print('Модель:', 'OK' if result.success else 'FAIL')
if not result.success:
    print('Errors:', result.errors)

# Load historical data for base rating
with Repository(db_path=DB_PATH) as repo:
    loader = ModelInputLoader(COMPANY_ID, repo, CONFIG_PATH, SCENARIO_NAME)
    historic, config = loader.load()

years = sorted(mr.years.keys())


## §2 Кредитные метрики

In [ ]:
with Repository(db_path=DB_PATH) as repo:
    loader = ModelInputLoader(COMPANY_ID, repo, CONFIG_PATH, SCENARIO_NAME)
    historic, config = loader.load()

METRICS_ITEMS = [
    ('Net Debt/EBITDA x',      lambda m: m.net_debt_ebitda),
    ('Debt/Equity x',          lambda m: m.debt_to_equity),
    ('Debt/Capital %',         lambda m: m.debt_to_capital * 100 if m.debt_to_capital else None),
    ('Interest Coverage x',    lambda m: m.interest_coverage),
    ('EBITDA Coverage x',      lambda m: m.ebitda_coverage),
    ('EBITDA Margin %',        lambda m: m.ebitda_margin * 100 if m.ebitda_margin else None),
    ('ROA %',                  lambda m: m.roa * 100 if m.roa else None),
    ('Current Ratio x',        lambda m: m.current_ratio),
    ('Cash/Debt %',            lambda m: m.cash_to_debt * 100 if m.cash_to_debt else None),
    ('FCF/Debt %',             lambda m: m.fcf_to_debt * 100 if m.fcf_to_debt else None),
]

rows = []
years = sorted(mr.years.keys())
for name, fn in METRICS_ITEMS:
    row = {'Метрика': name}
    for yr in years:
        m = CreditMetrics.from_year_state(mr.years[yr], yr)
        v = fn(m)
        row[yr] = f'{v:.1f}' if v is not None else 'N/A'
    rows.append(row)

df_metrics = pd.DataFrame(rows).set_index('Метрика')
print('Кредитные метрики:')
display(df_metrics)

## §3 Base / Forecast / Stress рейтинги

In [ ]:
# Rating from build_model: forecast + stress scenarios
forecast_r = result.rating_result

# Historical base rating
with Repository(db_path=DB_PATH) as repo:
    company_dir = ROOT / 'companies' / COMPANY_ID
    rating_runner = RatingRunner.from_project_yaml(COMPANY_ID, repo, company_dir)
    base_r = rating_runner.rate_historical(historic, save=False)

# Stress rating: use steel_downturn result from build_model
stress_r = None
if 'steel_downturn' in result.stress_results:
    sr = result.stress_results['steel_downturn']
    if sr.success:
        with Repository(db_path=DB_PATH) as repo:
            company_dir = ROOT / 'companies' / COMPANY_ID
            rr2 = RatingRunner.from_project_yaml(COMPANY_ID, repo, company_dir)
            stress_r = rr2.rate_model_result(sr.model_result, 'stress', save=False)

print('Рейтинги по сценариям:')
print()

all_ratings = [('FORECAST', forecast_r)]
if stress_r:
    all_ratings.append(('STRESS (steel_downturn)', stress_r))

for label, rr in all_ratings:
    if rr and rr.success:
        print(f'  {label}:')
        print(f'    {"Год":<6} {"Рейтинг":>8} {"Score":>6} {"IG/HY":>6} {"Lev":>6} {"Cov":>6} {"Prof":>6} {"Liq":>6}')
        for yr, r in sorted(rr.ratings.items()):
            ig = 'IG' if r.get('is_investment_grade') else 'HY'
            ss = r.get('sub_scores', {})
            print(f'    {yr:<6} {r["rating"]:>8} {r["score"]:>6.1f} {ig:>6}'
                  f' {ss.get("leverage",0):>6.1f} {ss.get("coverage",0):>6.1f}'
                  f' {ss.get("profitability",0):>6.1f} {ss.get("liquidity",0):>6.1f}')
        print()


## §4 Ковенанты

In [ ]:
cov_result = result.covenants_result

# Stress covenants: use steel_downturn
cov_stress = None
if 'steel_downturn' in result.stress_results:
    sr = result.stress_results['steel_downturn']
    if sr.success:
        with Repository(db_path=DB_PATH) as repo:
            company_dir = ROOT / 'companies' / COMPANY_ID
            chk = CovenantsChecker.from_project_yaml(COMPANY_ID, repo, company_dir)
            cov_stress = chk.check(sr.model_result, 'steel_downturn', save=False)

if cov_result:
    print('Ковенанты — BASE forecast:')
    print(cov_result.summary())

if cov_stress:
    print('Ковенанты — STRESS (steel_downturn):')
    print(cov_stress.summary())

# Comparison table
if cov_result and cov_stress:
    rows = []
    for r in cov_result.results:
        row = {
            'Год':      r.year,
            'Ковенант': r.covenant_name,
            'Base':     f'{r.value:.2f}' if r.value is not None else 'N/A',
            'Лимит':    f'{r.threshold:.2f}',
            'Base St':  'OK' if r.status=='ok' else 'WARN' if r.status=='warning' else 'BREACH',
        }
        stress_dict = {(s.year, s.covenant_name): s for s in cov_stress.results}
        sr_cov = stress_dict.get((r.year, r.covenant_name))
        if sr_cov:
            row['Stress'] = f'{sr_cov.value:.2f}' if sr_cov.value is not None else 'N/A'
            row['St St']  = 'OK' if sr_cov.status=='ok' else 'WARN' if sr_cov.status=='warning' else 'BREACH'
        rows.append(row)
    if rows:
        try:
            display(pd.DataFrame(rows))
        except:
            for row in rows[:10]:
                print(row)


## §5 Covenant Breach — влияние на рейтинг
Если база нарушает ковенант → авто-запуск `covenant_breach` сценария → рейтинг в стрессе.

In [ ]:
# Covenant breach auto-trigger demonstration
# In base scenario (strong financials) no breach expected.
# covenant_breach scenario runs automatically if orchestrator detects breach.

breaches = cov_result.breaches() if cov_result else []
print(f'Нарушений ковенантов (base): {len(breaches)}')

if 'covenant_breach' in result.stress_results:
    sr_cb = result.stress_results['covenant_breach']
    print('  Авто-запущен covenant_breach стресс-сценарий:')
    if sr_cb.success and sr_cb.comparison:
        yr0 = min(sr_cb.comparison.keys())
        c = sr_cb.comparison[yr0]
        print(f'    {yr0}: Rev={c["revenue_delta_pct"]:+.1f}% EBITDA={c["ebitda_delta_pct"]:+.1f}% NI={c["ni_delta_pct"]:+.1f}%')
    if sr_cb.success and hasattr(sr_cb, 'model_result') and sr_cb.model_result:
        with Repository(db_path=DB_PATH) as repo:
            company_dir = ROOT / 'companies' / COMPANY_ID
            rr_cb = RatingRunner.from_project_yaml(COMPANY_ID, repo, company_dir)
            r_cb = rr_cb.rate_model_result(sr_cb.model_result, 'covenant_breach', save=False)
        if r_cb.success:
            print(f'    Рейтинг при breach: {r_cb.worst_rating} (worst)')
else:
    print()
    print('  covenant_breach не авто-запущен (нарушений нет в базовом сценарии)')
    print()
    print('  Сценарий доступен для прямого запуска:')
    print('    stress_scenarios.yaml -> scenarios.covenant_breach')
    print('    Шоки: steel_price_hrc -15%, steel_ppi -10%, avg_rate +2pp, DSO/DIH +10%')


## §6 Итоговая таблица

In [ ]:
print('=' * 55)
print('КРЕДИТНЫЙ АНАЛИЗ: ' + COMPANY_ID.upper())
print('=' * 55)
print()

# Rating trajectory
if forecast_r and forecast_r.success:
    print('Рейтинговая динамика (Forecast):')
    print(f'  {"Год":<6} {"Рейтинг":>8} {"Score":>7} {"IG/HY":>6}')
    print('  ' + '-'*32)
    for yr, r in sorted(forecast_r.ratings.items()):
        ig = 'IG' if r.get('is_investment_grade') else 'HY'
        print(f'  {yr:<6} {r["rating"]:>8} {r["score"]:>7.1f} {ig:>6}')

if stress_r and stress_r.success:
    print()
    print('Рейтинговая динамика (steel_downturn stress):')
    print(f'  {"Год":<6} {"Рейтинг":>8} {"Score":>7} {"IG/HY":>6}')
    print('  ' + '-'*32)
    for yr, r in sorted(stress_r.ratings.items()):
        ig = 'IG' if r.get('is_investment_grade') else 'HY'
        print(f'  {yr:<6} {r["rating"]:>8} {r["score"]:>7.1f} {ig:>6}')

print()
if cov_result:
    nb_br = len(cov_result.breaches())
    fb    = cov_result.first_breach_year()
    print(f'  Ковенанты (base):   {nb_br} нарушений', end='')
    if fb: print(f'  первое: {fb}', end='')
    print()

if cov_stress:
    nb_br2 = len(cov_stress.breaches())
    fb2    = cov_stress.first_breach_year()
    print(f'  Ковенанты (stress): {nb_br2} нарушений', end='')
    if fb2: print(f'  первое: {fb2}', end='')
    print()

print()
if forecast_r and forecast_r.success:
    print(f'  Best forecast rating:  {forecast_r.best_rating}')
    print(f'  Worst forecast rating: {forecast_r.worst_rating}')
if stress_r and stress_r.success:
    print(f'  Worst stress rating:   {stress_r.worst_rating}')

print()
print('Covenant_breach сценарий:', 'covenant_breach' in result.stress_results and 'авто-запущен' or 'не авто-запущен (нарушений нет)')
